# Chapter 3 — 어텐션(Attention) : 코드 해설 노트북 (Python 초보용)

> `Chapter_3_Excercise_Attention.ipynb` 의 원본 코드 셀은 그대로 두고, 번호가 붙은 셀마다 바로 위에 설명을 넣은 해설판입니다.

## 이 챕터에서 하는 일 (한 문장)

**"문장 속 각 토큰이 다른 토큰들을 얼마나 참고할지 스스로 정하게 만드는 장치"**, 즉 어텐션을 직접 만듭니다. GPT 의 심장에 해당하는 부분입니다.

## 어텐션을 왜 쓸까?

"그 **동물**은 길을 건너지 않았다. **그것**은 너무 지쳤기 때문이다." 에서 "그것"이 무엇인지 알려면 앞의 "동물"을 봐야 합니다. Chapter 2 에서 만든 임베딩 벡터는 토큰 하나하나가 **따로따로**라서 이런 관계를 모릅니다. 어텐션은 각 토큰의 벡터에 **주변 토큰들의 정보를 필요한 만큼 섞어** 넣어, 문맥이 반영된 새 벡터(**문맥 벡터, context vector**)를 만들어 줍니다.

## 도서관 비유로 보는 Q, K, V

| 이름 | 비유 | 역할 |
|---|---|---|
| **Query (쿼리, 질문)** | 내가 찾고 싶은 것을 적은 검색어 | "나는 지금 이런 정보가 필요해" |
| **Key (키, 열쇠말)** | 책마다 붙은 색인/제목 | "나는 이런 정보를 갖고 있어" |
| **Value (밸류, 내용)** | 책의 실제 내용 | 실제로 가져올 정보 |

검색어(Q)와 색인(K)이 **잘 맞을수록** 그 책의 내용(V)을 **많이** 가져옵니다. 이것이 어텐션의 전부입니다.

```
 x  --W_query-->  Q --+
 x  --W_key---->  K --+--> scores = Q @ K^T --> mask --> softmax --> weights
 x  --W_value-->  V -------------------------------------------------+
                                                                     v
                                                  context = weights @ V
```

## 셀 구성

| 셀 | 내용 |
|---|---|
| 1 | `CausalAttention` — 미래를 못 보게 가린 어텐션 (헤드 1개) |
| 2 | 작은 예제 데이터로 실행해 보기 |
| 3 | `MultiHeadAttentionWrapper` — 어텐션 여러 개를 나란히 세운 단순한 멀티 헤드 |
| 4 | `MultiHeadAttention` — 한 번의 행렬 계산으로 처리하는 효율적인 멀티 헤드 (실제 GPT 에서 사용) |

---
## 셀 1 — `CausalAttention` 클래스 (⭐ 중요)

### 🎯 이 셀이 하는 일
**인과적(causal) 어텐션**을 구현합니다. "인과적"이란 **각 토큰이 자기 자신과 그 앞(과거)의 토큰만 볼 수 있고, 뒤(미래)의 토큰은 볼 수 없다**는 뜻입니다. GPT 는 다음 단어를 맞히는 모델인데, 정답인 뒤쪽 단어를 미리 볼 수 있으면 커닝이 되기 때문입니다.

### 먼저 : `nn.Module` 이란?
```python
class CausalAttention(nn.Module):
    def __init__(self, ...):
        super().__init__()
        ...
    def forward(self, x):
        ...
```
PyTorch 에서 신경망 부품은 모두 `nn.Module` 을 **상속**해서 만듭니다. 규칙은 두 가지입니다.

| 메서드 | 역할 | 비유 |
|---|---|---|
| `__init__` | 부품(레이어)들을 **준비**해 둠 | 요리 재료·도구 꺼내 놓기 |
| `forward` | 입력이 들어왔을 때 **실제 계산** 순서 | 조리 순서 |

- `super().__init__()` : 부모 클래스(`nn.Module`)의 준비 작업을 먼저 실행하라는 뜻입니다. **항상 첫 줄에 써야** 하는 정해진 문구입니다.
- 나중에 `ca(batch)` 처럼 객체를 함수처럼 부르면 PyTorch 가 알아서 `forward(batch)` 를 실행합니다.

### `__init__` 뜯어보기 — 재료 준비

**① Q, K, V 를 만드는 선형 레이어 3개**
```python
self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
```
`nn.Linear(d_in, d_out)` 은 길이 `d_in` 짜리 벡터를 길이 `d_out` 짜리 벡터로 바꿔 주는 가장 기본적인 레이어입니다(안에 **학습되는 숫자 표**가 들어 있습니다). 같은 입력 `x` 를 세 개의 서로 다른 표에 통과시켜 "질문용", "색인용", "내용용" 벡터를 따로 만듭니다.
`bias=qkv_bias` 는 더해 주는 상수(편향)를 쓸지 여부이며 기본은 `False` 입니다.

**② 드롭아웃**
```python
self.dropout = nn.Dropout(dropout)
```
학습 중에 숫자 일부를 **무작위로 0으로** 만드는 장치입니다. 모델이 특정 정보에만 지나치게 의존(과적합)하는 것을 막습니다. `dropout=0.1` 이면 10% 를 끕니다.

**③ 마스크(가림막) 만들기**
```python
self.register_buffer(
    'mask',
    torch.triu(torch.ones(context_length, context_length), diagonal=1)
)
```
- `torch.ones(n, n)` : 1로 가득 찬 n×n 표
- `torch.triu(..., diagonal=1)` : 대각선 **위쪽 삼각형만** 남기고 나머지는 0 으로 (triu = **tri**angle **u**pper)

`context_length=4` 일 때 :
```
            col: 0  1  2  3      (looking at)
 row 0 (tok 0)  [0, 1, 1, 1]
 row 1 (tok 1)  [0, 0, 1, 1]
 row 2 (tok 2)  [0, 0, 0, 1]
 row 3 (tok 3)  [0, 0, 0, 0]
```
행 = "보는 토큰", 열 = "보이는 토큰" 입니다. **1 = 가릴 자리(미래)**. 예) 1번 토큰(row 1)은 0, 1번은 볼 수 있고 2, 3번은 가려집니다.

- `register_buffer('mask', ...)` : 이 표를 `self.mask` 로 저장하되 "**학습으로 바뀌지는 않는** 고정 데이터"로 등록합니다. 이렇게 해 두면 모델을 GPU 로 옮길 때 같이 따라가고, 모델을 파일로 저장할 때도 함께 저장됩니다.

### `forward` 뜯어보기 — 계산 순서

**0단계 : 입력 모양 확인**
```python
b, num_tokens, d_in = x.shape
```
`x.shape` 는 `[배치 크기, 토큰 수, 벡터 길이]` 세 숫자입니다. 이를 세 변수에 한 번에 나눠 담았습니다(언패킹).

**1단계 : Q, K, V 만들기**
```python
keys    = self.W_key(x)
queries = self.W_query(x)
values  = self.W_value(x)      # 모두 [b, num_tokens, d_out]
```

**2단계 : 어텐션 스코어 (서로 얼마나 관련 있나)**
```python
attn_scores = queries @ keys.transpose(1, 2)
```
- `@` : 행렬 곱. 여기서는 "모든 Query 와 모든 Key 의 **내적**"을 한꺼번에 계산합니다. 내적은 두 벡터가 **비슷한 방향일수록 큰 값**이 나오므로 "관련도 점수"로 쓸 수 있습니다.
- `keys.transpose(1, 2)` : 1번 축과 2번 축을 맞바꿉니다. `[b, 토큰, d_out]` → `[b, d_out, 토큰]`. 행렬 곱은 **앞 행렬의 열 수 = 뒤 행렬의 행 수** 여야 해서 모양을 맞춰 주는 것입니다.
- 결과 모양 : `[b, 토큰, 토큰]` — "i 번 토큰이 j 번 토큰을 얼마나 주목하는가"를 담은 정사각형 점수표

**3단계 : 미래 가리기**
```python
attn_scores.masked_fill_(
    self.mask.bool()[:num_tokens, :num_tokens],
    -torch.inf
)
```
- `.bool()` : 1/0 을 True/False 로 바꿉니다.
- `[:num_tokens, :num_tokens]` : 실제 입력 길이만큼만 잘라 씁니다(입력이 최대 길이보다 짧을 수 있으므로).
- `masked_fill_(마스크, 값)` : 마스크가 True 인 자리를 그 값으로 덮어씁니다. 이름 끝의 **밑줄 `_`** 은 PyTorch 에서 "새로 만들지 않고 **제자리에서 바로 수정**한다"는 표시입니다.
- `-torch.inf` : 음의 무한대. **왜 0 이 아니라 −무한대일까요?** 바로 다음 단계인 softmax 가 e^(−∞) = 0 으로 만들어 주기 때문입니다. 점수 단계에서 0 을 넣으면 "보통 정도의 관련"으로 취급되어 여전히 정보가 샙니다.

**4단계 : 점수를 비율(확률)로 바꾸기**
```python
attn_weights = torch.softmax(
    attn_scores / keys.shape[-1]**0.5, dim=-1
)
```
- `torch.softmax(..., dim=-1)` : 각 행의 숫자들을 **모두 0~1 사이, 합이 1** 이 되게 바꿉니다. 큰 점수일수록 큰 비율을 받습니다. `dim=-1` 은 "마지막 축(= 각 행 안에서)"이라는 뜻입니다.
- `keys.shape[-1]**0.5` : `shape[-1]` 은 마지막 축 크기(= `d_out`), `**0.5` 는 제곱근. 벡터가 길수록 내적 값이 커지는데, 값이 너무 크면 softmax 가 한 곳에만 1 을 몰아줘서 학습이 어려워집니다. 그래서 √d_out 으로 **나눠서 크기를 눌러** 줍니다. 그래서 이 방식을 *scaled dot-product attention* 이라고 부릅니다.

가린 뒤 softmax 를 거친 결과의 예 :
```
 row 0  [1.00, 0.00, 0.00, 0.00]    token 0 sees only itself
 row 1  [0.45, 0.55, 0.00, 0.00]
 row 2  [0.30, 0.33, 0.37, 0.00]
 row 3  [0.20, 0.25, 0.27, 0.28]    each row sums to 1
```

**5단계 : 드롭아웃**
```python
attn_weights = self.dropout(attn_weights)
```

**6단계 : 문맥 벡터 만들기**
```python
context_vec = attn_weights @ values
return context_vec
```
비율(가중치)대로 Value 들을 섞습니다. 위 예에서 2번 토큰의 새 벡터는 `0.30×V0 + 0.33×V1 + 0.37×V2` 가 됩니다. 결과 모양은 `[b, 토큰 수, d_out]`.

### 모양(shape) 변화 한눈에 보기
```
 x            [b, T, d_in]
 Q, K, V      [b, T, d_out]
 attn_scores  [b, T, T]
 attn_weights [b, T, T]
 context_vec  [b, T, d_out]        (T = num_tokens)
```

> 이 셀은 클래스 정의만 하므로 출력이 없습니다.

In [2]:
# 1
### 중요: 이 코드는 인과적 어텐션 메커니즘을 구현한 PyTorch 모듈입니다.
import torch
import torch.nn as nn

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        """
        Args:
            d_in: 입력 벡터의 차원 크기
            d_out: 출력(및 쿼리/키/밸류) 벡터의 차원 크기
            context_length: 모델이 한 번에 처리할 수 있는 최대 문맥 길이 (토큰 수)
            dropout: 드롭아웃 확률
            qkv_bias: 선형 레이어에 편향(bias)을 사용할지 여부
        """
        super().__init__()
        
        # 1. 쿼리(Query), 키(Key), 밸류(Value)를 만들기 위한 선형 투영 레이어 정의
        # 입력 벡터(d_in)를 각각의 목적에 맞는 벡터(d_out)로 변환합니다.
        # TODO: Q/K/V 투영 레이어 타입과 인자를 채우세요.
        # 힌트: 입력 d_in, 출력 d_out을 사용하는 Linear 레이어를 선언하세요.
        # self.W_query = nn.????(????, ????, bias=qkv_bias)
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        # self.W_key   = nn.????(????, ????, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        # self.W_value = nn.????(????, ????, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        # 2. 과적합(Overfitting) 방지를 위한 드롭아웃 설정
        self.dropout = nn.Dropout(dropout)

        # 3. 인과적 마스크(Causal Mask) 생성 및 버퍼 등록
        # 'register_buffer'를 사용하면 역전파(학습) 대상은 아니지만, 모델의 상태(state_dict)로 저장됩니다.
        # torch.triu(..., diagonal=1): 대각선(0) 위쪽 삼각형 부분만 1로 채웁니다.
        # 즉, '미래의 정보' 위치에 1을 표시하여 나중에 가릴(masking) 준비를 합니다.
        self.register_buffer(
            'mask', 
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        # x.shape: [배치 크기(b), 토큰 개수(num_tokens), 입력 차원(d_in)]
        b, num_tokens, d_in = x.shape 

        # 입력 x를 통과시켜 현재 시점의 관심사(Query), 검색 대상(Key), 정보 내용(Value)을 추출합니다.
        # region [Q, K, V 벡터 계산]
        # TODO: Q, K, V 계산 호출 대상을 채우세요.
        # 힌트: W_query, W_key, W_value를 각각 x에 적용하면 됩니다.
        # keys = self.????(????)        # Shape: [b, num_tokens, d_out]
        keys = self.W_key(x)        # Shape: [b, num_tokens, d_out]
        # queries = self.?????(????)   # Shape: [b, num_tokens, d_out]
        queries = self.W_query(x)   # Shape: [b, num_tokens, d_out]
        # values = self.?????(????)    # Shape: [b, num_tokens, d_out]
        values = self.W_value(x)    # Shape: [b, num_tokens, d_out]
        # endregion

        # Query와 Key의 내적(Dot Product)을 통해 각 토큰 간의 관련성을 구합니다.
        # keys.transpose(1, 2): 행렬 곱을 위해 차원을 뒤집습니다. (d_out 차원끼리 곱해짐)
        # region [어텐션 스코어(유사도) 계산]
        # TODO: 어텐션 스코어 행렬곱 피연산자를 채우세요.
        # 힌트: queries와 keys.transpose(1, 2)를 곱해 유사도 점수를 구합니다.
        # attn_scores = ???? @ ????.transpose(1, 2) 
        attn_scores = queries @ keys.transpose(1, 2) 
        #endregion

        # mask가 1인 위치(미래 시점의 토큰들)를 -무한대(-inf)로 채웁니다.
        # 이렇게 하면 나중에 Softmax를 거칠 때 확률이 0이 되어, 미래 정보를 참조하지 못하게 됩니다.
        # [:num_tokens, :num_tokens]: 입력 길이가 context_length보다 짧을 때를 대비해 크기를 맞춥니다.
        # region [인과적 마스킹 (Masking) - 미래 정보 차단]
        # TODO: 미래 시점의 어텐션 스코어를 가릴 값을 채우세요.
        # 힌트: Softmax를 통과하면 0이 되도록 마이너스 무한대(-torch.inf)를 입력하세요.
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], 
            # ????
            -torch.inf
        ) 
        # endregion

        # 스케일링(/ keys.shape[-1]**0.5): 차원이 커질수록 내적 값이 커져 기울기 소실이 오는 것을 방지합니다.
        # Softmax: 점수를 확률(0~1 사이, 합은 1)로 변환합니다.
        # region [어텐션 가중치(Weights) 계산 및 스케일링]
        # TODO: 어텐션 가중치 함수명을 채우세요.
        # 힌트: 스케일 적용 후 softmax를 사용해 확률로 변환하세요.
        # attn_weights = torch.????(
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        # endregion

        # 계산된 가중치 중 일부를 무작위로 0으로 만들어 모델이 특정 토큰에만 의존하는 것을 막습니다.
        # region [드롭아웃 적용]
        attn_weights = self.dropout(attn_weights)
        # endregion

        # 어텐션 가중치(확률)를 기반으로 Value(정보)들을 가중 합산합니다.
        # 결과적으로 "현재 토큰과 관련이 깊은 과거 토큰들의 정보"가 진하게 섞인 벡터가 됩니다.
        # region [문맥 벡터(Context Vector) 생성]
        # TODO: 컨텍스트 벡터 계산 피연산자를 채우세요.
        # 힌트: attn_weights와 values를 곱해 최종 컨텍스트 벡터를 얻습니다.
        # context_vec = attn_weights @ ????
        context_vec = attn_weights @ values
        # endregion
        
        return context_vec


---
## 셀 2 — 작은 예제로 `CausalAttention` 실행

### 🎯 이 셀이 하는 일
"Your journey starts with one step" 여섯 단어를 각각 숫자 3개짜리 벡터로 표현한 장난감 데이터를 만들고, 셀 1의 어텐션에 넣어 결과 모양을 확인합니다.

### 코드 뜯어보기

```python
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your
   [0.55, 0.87, 0.66], # journey
   ...
   [0.05, 0.80, 0.55]] # step
)
```
6행 3열 텐서입니다. 한 행이 토큰 하나의 임베딩 벡터라고 생각하면 됩니다. (실제로는 Chapter 2 의 임베딩 레이어가 만들어 주지만, 여기서는 손으로 적었습니다.)

```python
d_in = inputs.shape[1]   # 3
d_out = 2
```
`inputs.shape` 는 `[6, 3]` 이고 `shape[1]` 은 그중 두 번째 값 3 입니다(번호는 0부터).

```python
batch = torch.stack((inputs, inputs), dim=0)
```
`torch.stack` 은 같은 모양의 텐서를 **새 축을 만들어 쌓습니다.** `[6, 3]` 두 장을 쌓아 `[2, 6, 3]` 이 됩니다. 똑같은 문장 2개로 이루어진 "배치"를 흉내 낸 것입니다. 괄호가 두 겹인 이유는 `(inputs, inputs)` 가 하나의 튜플 인자이기 때문입니다.

```python
context_length = batch.shape[1]     # 6
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
```
- `CausalAttention(3, 2, 6, 0.0)` : 객체를 만듭니다(`__init__` 실행). 드롭아웃은 0.0 → 끔.
- `ca(batch)` : 객체를 함수처럼 부르면 `forward(batch)` 가 실행됩니다. (빈칸 문제의 정답이 `ca`)

### 실행 결과 읽기
```
context_length: 6
tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         ...
context_vecs.shape: torch.Size([2, 6, 2])
```
- 모양 `[2, 6, 2]` = 문장 2개 × 토큰 6개 × 숫자 2개(`d_out`). 입력 `[2, 6, 3]` 에서 **마지막 축만 3 → 2** 로 바뀌었습니다.
- 두 문장이 똑같으니 결과도 위아래 블록이 똑같습니다.
- `grad_fn=...` 은 "이 값이 어떤 계산으로 만들어졌는지 PyTorch 가 기억하고 있다"는 표시이며, 나중에 학습(역전파)에 쓰입니다.

In [3]:
# 2
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

d_in = inputs.shape[1]   # 입력 차원 (d=3)
d_out = 2                # Q, K, V의 출력 차원 (d=2)

batch = torch.stack((inputs, inputs), dim=0)
# --- 실행 예시 ---
torch.manual_seed(123)

# 가정: batch 변수가 이미 정의되어 있다고 가정 (예: b=2, num_tokens=6, d_in=...)
# context_length는 모델이 허용하는 최대 길이이므로, 현재 배치의 길이와 같거나 더 길게 설정합니다.
context_length = batch.shape[1] 

print("context_length:", context_length)

# TODO: 어텐션 모듈 호출 대상을 채우세요.
# 힌트: 직전에 생성한 CausalAttention 인스턴스(ca)를 호출하면 됩니다.
ca = CausalAttention(d_in, d_out, context_length, 0.0)
# context_vecs = ????(batch)
context_vecs = ca(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

context_length: 6
tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


---
## 셀 3 — `MultiHeadAttentionWrapper` (단순한 멀티 헤드)

### 🎯 이 셀이 하는 일
어텐션을 **여러 개(여러 "헤드") 나란히** 돌리고 결과를 옆으로 이어 붙입니다.

### 왜 여러 개가 필요할까?
어텐션 하나는 한 가지 관점으로만 문장을 봅니다. 여러 개를 두면 어떤 헤드는 "주어–동사 관계", 다른 헤드는 "바로 앞 단어", 또 다른 헤드는 "대명사가 가리키는 대상"처럼 **서로 다른 관점**을 동시에 배울 수 있습니다. 여러 명이 같은 글을 각자 다른 색 형광펜으로 읽는 것과 비슷합니다.

### 코드 뜯어보기
```python
self.heads = nn.ModuleList(
    [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
     for _ in range(num_heads)]
)
```
- 대괄호 안은 **리스트 컴프리헨션**입니다. "`CausalAttention` 을 `num_heads` 번 만들어 리스트에 담아라"는 뜻이며, 반복 변수를 쓰지 않을 때는 관례로 `_` 를 씁니다.
- `nn.ModuleList` : 일반 리스트와 거의 같지만, PyTorch 가 안에 든 레이어들을 **모델의 부품으로 인식**하게 해 줍니다(그래야 학습·저장·GPU 이동이 됩니다).

```python
def forward(self, x):
    return torch.cat([head(x) for head in self.heads], dim=-1)
```
- `[head(x) for head in self.heads]` : 각 헤드에 같은 입력을 넣어 결과를 리스트로 모읍니다.
- `torch.cat(..., dim=-1)` : 마지막 축 방향으로 **이어 붙입니다.** (`stack` 은 새 축을 만들고, `cat` 은 기존 축을 따라 길게 붙인다는 차이가 있습니다.)

```
 head 1 output  [2, 6, 2]
 head 2 output  [2, 6, 2]
 cat(dim=-1) -> [2, 6, 4]
```

### 실행 결과 읽기
`context_vecs.shape: torch.Size([2, 6, 4])` — 헤드 2개 × `d_out` 2 = 4.
출력 숫자의 **앞 두 열이 셀 2의 결과와 똑같습니다**(-0.4519, 0.2216 …). 시드가 같아서 첫 번째 헤드가 셀 2의 어텐션과 똑같이 만들어졌기 때문입니다. 뒤 두 열이 두 번째 헤드의 결과입니다.

### 이 방식의 아쉬운 점
헤드를 `for` 로 **하나씩 차례로** 계산하므로 느립니다. 셀 4에서 이를 개선합니다.

In [4]:
# 3
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) 
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

torch.manual_seed(123)

context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


---
## 셀 4 — `MultiHeadAttention` (효율적인 멀티 헤드, ⭐ 실제 GPT 에서 사용)

### 🎯 이 셀이 하는 일
셀 3과 같은 일을 하되, 헤드별로 레이어를 따로 두지 않고 **큰 Q/K/V 를 한 번에 만든 뒤 모양만 바꿔서 헤드 수만큼 쪼개** 한꺼번에 계산합니다. Chapter 4 이후로는 계속 이 클래스를 씁니다.

### 셀 3과의 차이

| | 셀 3 (Wrapper) | 셀 4 (이 클래스) |
|---|---|---|
| 계산 방식 | 헤드를 하나씩 순서대로 | 모든 헤드를 행렬 연산 한 번에 |
| `d_out` 의 뜻 | 헤드 **하나**의 출력 크기 | **전체** 출력 크기 (헤드들이 나눠 가짐) |
| 최종 출력 크기 | `d_out × num_heads` | `d_out` |

### `__init__` 뜯어보기
```python
assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"
self.head_dim = d_out // num_heads
```
- `%` 는 나머지, `//` 는 몫(정수 나눗셈)입니다.
- 전체 크기 `d_out` 을 헤드들이 똑같이 나눠 가져야 하므로 나누어떨어지는지 검사합니다. 예) GPT-2 : 768 ÷ 12 헤드 = 헤드당 64.
- 줄 끝의 `\` 는 "이 줄이 다음 줄로 이어진다"는 표시입니다.

```python
self.out_proj = nn.Linear(d_out, d_out)
```
헤드들의 결과를 합친 뒤 한 번 더 섞어 주는 마무리 레이어입니다. 나머지(`W_query` 등, `dropout`, `mask`)는 셀 1과 같습니다.

### `forward` 뜯어보기

**① Q, K, V 를 크게 한 번에**
```python
keys = self.W_key(x)        # [b, T, d_out]
```

**② 헤드 수만큼 쪼개기 — `view`**
```python
keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
```
`view` 는 **숫자는 그대로 두고 모양만** 바꿉니다. 마지막 축 `d_out` 을 `num_heads × head_dim` 으로 접은 것입니다.
```
 [b, T, d_out]  ->  [b, T, num_heads, head_dim]
 e.g. [2, 6, 2] ->  [2, 6, 2, 1]
```

**③ 축 순서 바꾸기 — `transpose`**
```python
keys = keys.transpose(1, 2)     # [b, num_heads, T, head_dim]
```
1번 축(토큰)과 2번 축(헤드)을 맞바꿔 **헤드 축을 앞으로** 보냅니다. 이렇게 하면 뒤의 두 축 `[T, head_dim]` 이 "헤드 하나의 Q/K/V" 가 되어, 셀 1에서 하던 계산을 헤드마다 독립적으로 **동시에** 할 수 있습니다. (PyTorch 의 `@` 는 뒤의 두 축끼리만 행렬 곱을 하고, 앞의 축들은 "묶음"으로 취급합니다.)

**④ 여기서부터는 셀 1과 동일**
```python
attn_scores = queries @ keys.transpose(2, 3)          # [b, heads, T, T]
mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
attn_scores.masked_fill_(mask_bool, -torch.inf)
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
attn_weights = self.dropout(attn_weights)
```
축이 하나 늘었기 때문에 `transpose(1, 2)` 가 `transpose(2, 3)` 으로 바뀐 것만 다릅니다. `[T, T]` 모양의 마스크는 브로드캐스팅으로 모든 배치·모든 헤드에 똑같이 적용됩니다.

**⑤ 다시 합치기**
```python
context_vec = (attn_weights @ values).transpose(1, 2)                  # [b, T, heads, head_dim]
context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out) # [b, T, d_out]
context_vec = self.out_proj(context_vec)
```
- `transpose(1, 2)` : ③에서 바꿨던 축 순서를 **원래대로** 되돌립니다.
- `.contiguous()` : `transpose` 를 하면 메모리 안의 숫자 배열 순서와 겉모양이 어긋나는데, `view` 는 그 상태에서는 쓸 수 없습니다. `.contiguous()` 가 메모리를 새 순서대로 다시 정리해 줍니다. "**transpose 뒤에 view 를 쓰려면 contiguous 가 필요하다**"로 기억하면 충분합니다.
- `view(b, T, d_out)` : 헤드 축과 `head_dim` 축을 다시 하나로 펼칩니다 = 헤드들의 결과를 옆으로 이어 붙인 것과 같습니다.

### 모양 변화 한눈에 보기
```
 x                         [b, T, d_in]
 W_query / W_key / W_value [b, T, d_out]
 view                      [b, T, heads, head_dim]
 transpose(1,2)            [b, heads, T, head_dim]
 Q @ K^T                   [b, heads, T, T]
 weights @ V               [b, heads, T, head_dim]
 transpose(1,2)            [b, T, heads, head_dim]
 view                      [b, T, d_out]
 out_proj                  [b, T, d_out]
```

### 아래쪽 실행 코드
```python
batch_size, context_length, d_in = batch.shape      # 2, 6, 3
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
```

### 실행 결과 읽기
`context_vecs.shape: torch.Size([2, 6, 2])` — 셀 3은 `[2, 6, 4]` 였지만 여기서는 `d_out=2` 를 헤드 2개가 **1씩 나눠 가져서** 최종 크기가 2 입니다. 숫자가 셀 3과 다른 것은 구조(레이어 구성과 `out_proj`)가 달라 무작위 초기값이 다르게 배정되기 때문이며, 정상입니다.

In [5]:
# 4
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds `context_length`, 
        # this will result in errors in the mask creation further below. 
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs  
        # do not exceed `context_length` before reaching this forward method.

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


<img src="llm_from_scratch/images/llm_from_scratch/ch03_compressed/26.webp" width="600px">

## Simulation
- https://poloclub.github.io/transformer-explainer/
- https://bbycroft.net/llm


---
## 📌 Chapter 3 정리

**어텐션 6단계** (셀 1의 `forward`)
```
 1. Q, K, V = W_query(x), W_key(x), W_value(x)
 2. scores  = Q @ K^T                 how related are tokens?
 3. scores[future] = -inf             hide the future (causal mask)
 4. weights = softmax(scores / sqrt(d))   each row sums to 1
 5. weights = dropout(weights)
 6. context = weights @ V             mix values by weights
```

**꼭 기억할 3가지**
1. 어텐션 = "관련 있는 토큰의 정보를 **비율대로 섞어** 새 벡터를 만드는 것"
2. GPT 는 **미래 토큰을 가린다**(causal mask). −무한대로 채운 뒤 softmax 하면 그 자리가 0 이 된다.
3. 멀티 헤드 = 여러 관점으로 동시에 보기. 실전에서는 `view` + `transpose` 로 한 번에 계산한다.

바로 위 **Simulation** 칸의 링크(transformer-explainer, bbycroft.net/llm)를 열어 보면 이 과정을 움직이는 그림으로 볼 수 있습니다. 다음 노트북에서는 실제 GPT-2 의 어텐션이 어디를 보는지 그림으로 확인합니다.